# Maman 14 - Deep Learning 
**Name:** Lian Sigawi  
**ID:** 318814746

## Question 1: SplitLinear Layer
**Implementation (Sections A & B)**

The following class implements a custom SplitLinear layer that divides the input tensor into two halves along the feature dimension. It processes each half through a shared linear layer and a ReLU activation before concatenating them back together

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader


class SplitLinear(nn.Module):
    """
    Implementation of a SplitLinear layer that splits the input, 
    applies a shared linear transformation + ReLU, and concatenates the results.
    """
    def __init__(self, num_features):
        """
        Constructor (Section B): Initializes parameters and sets shared layers.
        """
        super(SplitLinear, self).__init__()
        
        # Ensure the features can be split equally (Section A)
        if num_features % 2 != 0:
            raise ValueError("num_features must be even for a 50/50 split.") 
            
        self.num_features = num_features
        self.half_features = num_features // 2
        
        # Shared linear layer for both halves (Section A)
        self.shared_linear = nn.Linear(self.half_features, self.half_features) 
        # Explicit parameter initialization (Section B)
        self.init_weights()

    def init_weights(self):
        """
        Initializes weights using He (Kaiming) initialization for ReLU compatibility,
        and sets biases to zero to maintain a neutral starting point (Section B).
        """
        # He initialization preserves variance across layers, preventing vanishing/exploding gradients 
        nn.init.kaiming_normal_(self.shared_linear.weight, mode='fan_in', nonlinearity='relu')
        
        # Biases initialized to 0 to avoid arbitrary shifts before training (Section B) 
        if self.shared_linear.bias is not None:
            nn.init.constant_(self.shared_linear.bias, 0)

    def forward(self, x):
        """
        Forward Pass (Section A): Splits, processes, and concatenates the tensor.
        """
        # 1. Split input tensor X (N x M) into two equal halves (Section A) 
        x1 = x[:, :self.half_features]
        x2 = x[:, self.half_features:]
        
        # 2+3. Pass both halves through the shared linear layer and apply ReLU (Section A) 
        out1 = F.relu(self.shared_linear(x1))
        out2 = F.relu(self.shared_linear(x2))
        
        # 4+5. Concatenate results back to the original dimension N x M (Section A) 
        return torch.cat([out1, out2], dim=1)

**Demonstration & Intermediate Steps (Section C)**

This section demonstrates the layer's operation by feeding it an example input and printing the results of each computational stage

In [ ]:
# Parameters for demonstration
N = 2  # Batch size
M = 4  # Number of features (must be even)

# Initialize the layer (Section B)
model = SplitLinear(num_features=M)
model.eval() 

# Example input tensor (N x M)
x = torch.tensor([[1.0, -2.0, 3.0, 4.0], 
                  [-1.0, 0.5, -3.0, 2.0]])

print("--- Step 1: Input Tensor (X) ---")
print(f"Data:\n{x}\nShape: {x.shape}\n")

# Intermediate logic (Section A)
half_m = M // 2

# Splitting logic 
x1 = x[:, :half_m]
x2 = x[:, half_m:]
print("--- Step 2: Split Inputs ---")
print(f"Half 1:\n{x1}\nHalf 2:\n{x2}\n")

# Shared Linear transformation 
out1_linear = model.shared_linear(x1)
out2_linear = model.shared_linear(x2)
print("--- Step 3: After Shared Linear Layer (Before ReLU) ---")
print(f"Output 1:\n{out1_linear.detach()}\nOutput 2:\n{out2_linear.detach()}\n")

# ReLU Activation
out1_relu = F.relu(out1_linear)
out2_relu = F.relu(out2_linear)
print("--- Step 4: After ReLU Activation ---")
print(f"ReLU 1:\n{out1_relu.detach()}\nReLU 2:\n{out2_relu.detach()}\n")

# Final Concatenation 
y = torch.cat([out1_relu, out2_relu], dim=1)
print("--- Step 5: Final Concatenated Output (Y) ---")
print(f"Data:\n{y.detach()}\nShape: {y.shape}")

--- Step 1: Input Tensor (X) ---
tensor([[ 1.0000, -2.0000,  3.0000,  4.0000],
        [-1.0000,  0.5000, -3.0000,  2.0000]])
Shape: torch.Size([2, 4])

--- Step 2: Split Inputs ---
Half 1:
tensor([[ 1.0000, -2.0000],
        [-1.0000,  0.5000]])
Half 2:
tensor([[ 3.,  4.],
        [-3.,  2.]])

--- Step 3: After Shared Linear Layer (Before ReLU) ---
Output 1:
tensor([[-1.2749, -2.3753],
        [-0.0164,  0.8339]])
Output 2:
tensor([[4.7838, 3.1501],
        [0.3812, 3.0155]])

--- Step 4: After ReLU Activation ---
ReLU 1:
tensor([[0.0000, 0.0000],
        [0.0000, 0.8339]])
ReLU 2:
tensor([[4.7838, 3.1501],
        [0.3812, 3.0155]])

--- Step 5: Final Concatenated Output (Y) ---
tensor([[0.0000, 0.0000, 4.7838, 3.1501],
        [0.0000, 0.8339, 0.3812, 3.0155]])
Shape: torch.Size([2, 4])


## Question 2: DropNorm Layer 
**Implementation (Sections A, B, & C)**

The DropNorm layer combines feature dropout and normalization. During training, it randomly zeroes out 50% of the features and normalizes the remaining ones.

In [ ]:
class DropNorm(nn.Module):
    """
    Custom layer performing 50% feature dropout followed by normalization
    and learnable scaling (gamma) and shifting (beta).
    """
    def __init__(self, num_features, eps=1e-5):
        """
        Constructor (Section C): Sets up learnable parameters.
        """
        super(DropNorm, self).__init__()
        self.num_features = num_features
        self.eps = eps
        
        # Learnable parameters (Section A) 
        self.gamma = nn.Parameter(torch.ones(num_features))
        self.beta = nn.Parameter(torch.zeros(num_features))
        
        self.init_parameters()

    def init_parameters(self):
        """
        Initializes parameters so the layer starts as an identity transform (Section C).
        """
        nn.init.constant_(self.gamma, 1)
        nn.init.constant_(self.beta, 0) 

    def forward(self, x):
        orig_shape = x.shape
        x_flat = x.view(x.shape[0], -1)
        N, M = x_flat.shape
        
        if self.training:
            # Randomly select 50% of features to drop (Section A) 
            perm = torch.randperm(M, device=x.device)
            keep_indices = perm[:M//2]
            
            mask = torch.zeros(M, device=x.device)
            mask[keep_indices] = 1
            
            # Calculate mean and variance for active features (Section A)
            active_features = x_flat[:, keep_indices]
            mu = active_features.mean(dim=1, keepdim=True)
            var = active_features.var(dim=1, keepdim=True, unbiased=False)
            
            #  Normalize and apply learnable transformation (Section A) 
            x_hat = (x_flat - mu) / torch.sqrt(var + self.eps)
            y = self.gamma * x_hat + self.beta
            
            # Apply dropout mask 
            return (y * mask).view(orig_shape)
            
        else:
            # Inference Mode (Section B): No dropout, use all information
            mu = x_flat.mean(dim=1, keepdim=True)
            var = x_flat.var(dim=1, keepdim=True, unbiased=False)
            
            x_hat = (x_flat - mu) / torch.sqrt(var + self.eps)
            
            # Adjust scaling to account for the training dropout factor 
            y = (self.gamma * x_hat + self.beta) * 0.5
            return y.view(orig_shape)

**Section D: Training with PyTorch Built-in Layers**

This section implements and trains a deep neural network using PyTorch's native BatchNorm1d and Dropout layers. This serves as a baseline for performance and convergence comparison.

In [ ]:

#  Device Setup: Use GPU acceleration if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 2. Data Loading: Fetching and normalizing the Fashion-MNIST dataset
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)) # Standard image normalization
])

train_set = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
test_set = datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
test_loader = DataLoader(test_set, batch_size=64, shuffle=False)

# Section D: Model definition using standard PyTorch built-in layers
class BuiltInModel(nn.Module):
    def __init__(self):
        super(BuiltInModel, self).__init__()
        self.flatten = nn.Flatten()
        self.network = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.BatchNorm1d(512),   # Standard normalization layer
            nn.ReLU(),
            nn.Dropout(0.5),       # Dropout layer with 50% probability
            
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.5),
            
            nn.Linear(256, 10)     # Output layer for 10 distinct classes

        )

    def forward(self, x):
        x = self.flatten(x)
        return self.network(x)

# Training Configuration: Initializing model, loss function, and optimizer
model = BuiltInModel().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Model Training: Standard loop for parameter optimization
def train_model(model, loader, epochs=5):
    model.train()
    for epoch in range(epochs):
        running_loss = 0.0
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            # Log average loss per epoch
        print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss/len(loader):.4f}")
        
# Execute training
print("Starting training with Built-in layers...")
train_model(model, train_loader)

Using device: cpu


100.0%
100.0%
100.0%
100.0%


Starting training with Built-in layers...
Epoch 1/5, Loss: 0.5455
Epoch 2/5, Loss: 0.4318
Epoch 3/5, Loss: 0.3953
Epoch 4/5, Loss: 0.3768
Epoch 5/5, Loss: 0.3612


**Section E: Training with Custom DropNorm Layer**

This section replaces the standard layers with the custom DropNorm implementation. The model architecture remains identical to ensure a fair comparison of the normalization and dropout mechanism

In [ ]:
# Model definition using the custom DropNorm layer (Section E)
class CustomDropNormModel(nn.Module):
    def __init__(self):
        super(CustomDropNormModel, self).__init__()
        self.flatten = nn.Flatten()
        self.network = nn.Sequential(
            nn.Linear(28*28, 512),
            DropNorm(512),   # Custom implementation combining Dropout and Normalization
            nn.ReLU(),
            
            nn.Linear(512, 256),
            DropNorm(256),   # Reuse of the custom layer
            nn.ReLU(),
            
            nn.Linear(256, 10)
        )

    def forward(self, x):
        x = self.flatten(x)
        return self.network(x)

# Initialization for custom model training
custom_model = CustomDropNormModel().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(custom_model.parameters(), lr=0.001)

# Training execution
print("Starting training with Custom DropNorm layer...")
train_model(custom_model, train_loader, epochs=5)

Starting training with Custom DropNorm layer...
Epoch 1/5, Loss: 0.6679
Epoch 2/5, Loss: 0.4954
Epoch 3/5, Loss: 0.4531
Epoch 4/5, Loss: 0.4278
Epoch 5/5, Loss: 0.4127


In [ ]:
def evaluate_accuracy(trained_model, data_loader):
    trained_model.eval()
    correct = 0
    total = 0

    with torch.inference_mode():
        for images, labels in data_loader:
            images = images.to(device)
            labels = labels.to(device)

            predictions = trained_model(images).argmax(dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    return 100 * correct / total


baseline_accuracy = evaluate_accuracy(model, test_loader)
dropnorm_accuracy = evaluate_accuracy(custom_model, test_loader)

print(f"BatchNorm + Dropout test accuracy: {baseline_accuracy:.2f}%")
print(f"DropNorm test accuracy:            {dropnorm_accuracy:.2f}%")